# 02. 데이터 전처리 (Preprocessing)
**Credit Risk Dataset — 대출 금리 예측 프로젝트**

---
### 처리 단계
1. 원본 데이터 로드
2. 결측치 처리 (삭제 / 중앙값 대체)
3. 이상치 제거
4. 범주형 변수 인코딩 (LabelEncoder)
5. 전처리 결과 확인 및 저장

---
## 0. 환경 설정

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110

ROOT           = os.path.abspath('..')
RAW_PATH       = os.path.join(ROOT, 'credit_risk_dataset.csv')
PROCESSED_DIR  = os.path.join(ROOT, 'data', 'processed')
PROCESSED_PATH = os.path.join(PROCESSED_DIR, 'credit_risk_cleaned.csv')

os.makedirs(PROCESSED_DIR, exist_ok=True)
print('설정 완료')

---
## 1. 원본 데이터 로드

In [ ]:
df_raw = pd.read_csv(RAW_PATH)
print(f'원본 데이터: {df_raw.shape[0]:,}행 x {df_raw.shape[1]}컬럼')
df_raw.head(3)

In [ ]:
print('컬럼 타입 요약:')
print(df_raw.dtypes.to_frame('dtype').assign(non_null=df_raw.notna().sum()))

---
## 2. 결측치 처리

In [ ]:
df = df_raw.copy()

miss_before = df.isnull().sum()
print('처리 전 결측치:')
print(miss_before[miss_before > 0])

In [ ]:
# loan_int_rate 결측 → 행 삭제 (목표변수)
before = len(df)
df     = df.dropna(subset=['loan_int_rate']).copy()
print(f'loan_int_rate 결측 행 삭제: {before:,} → {len(df):,}행 ({before-len(df):,}행 제거)')

In [ ]:
# person_emp_length 결측 → 중앙값 대체
median_emp = df['person_emp_length'].median()
miss_cnt   = df['person_emp_length'].isna().sum()
df['person_emp_length'] = df['person_emp_length'].fillna(median_emp)
print(f'person_emp_length 결측 {miss_cnt}개 → 중앙값 {median_emp}으로 대체')
print(f'처리 후 결측치 수: {df.isnull().sum().sum()}')

---
## 3. 이상치 제거

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['person_age'], bins=40, color='#4C72B0', edgecolor='white', linewidth=0.3)
axes[0].axvline(100, color='red', linestyle='--', lw=1.5, label='이상치 기준 (>100)')
axes[0].set_title('나이 분포 (전처리 전)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('나이 (세)')
axes[0].legend()

axes[1].hist(df['person_emp_length'], bins=40, color='#55A868', edgecolor='white', linewidth=0.3)
axes[1].axvline(60, color='red', linestyle='--', lw=1.5, label='이상치 기준 (>60)')
axes[1].set_title('재직기간 분포 (전처리 전)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('재직기간 (년)')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'나이 >100 이상치: {(df["person_age"] > 100).sum()}건')
print(f'재직기간 >60 이상치: {(df["person_emp_length"] > 60).sum()}건')

In [ ]:
before = len(df)
df     = df[df['person_age'] <= 100].copy()
df     = df[df['person_emp_length'] <= 60].copy()
print(f'이상치 제거: {before:,} → {len(df):,}행 ({before-len(df):,}행 제거)')

---
## 4. 범주형 변수 인코딩 (LabelEncoder)

In [ ]:
CAT_COLS = ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']

print('인코딩 전 고유값:')
for col in CAT_COLS:
    print(f'  {col}: {sorted(df[col].unique())}')

In [ ]:
encoding_maps = {}
for col in CAT_COLS:
    le              = LabelEncoder()
    df[col]         = le.fit_transform(df[col])
    encoding_maps[col] = dict(zip(le.classes_, le.transform(le.classes_)))

print('인코딩 매핑 (LabelEncoder — 알파벳 순):')
for col, mapping in encoding_maps.items():
    print(f'  {col}: {mapping}')

---
## 5. 전처리 결과 확인 및 저장

In [ ]:
print(f'최종 데이터: {df.shape[0]:,}행 x {df.shape[1]}컬럼')
print(f'결측치 잔여: {df.isnull().sum().sum()}')
df.head()

In [ ]:
df.describe().T.round(3)

In [ ]:
print('='*45)
print('전처리 단계별 데이터 크기 변화')
print('='*45)
print(f'원본 데이터:           {df_raw.shape[0]:>7,}행')
print(f'loan_int_rate 삭제 후: {df_raw.shape[0] - df_raw["loan_int_rate"].isna().sum():>7,}행')
print(f'이상치 제거 후:        {df.shape[0]:>7,}행')

In [ ]:
df.to_csv(PROCESSED_PATH, index=False)
print(f'저장 완료: {PROCESSED_PATH}')
print(f'파일 크기: {os.path.getsize(PROCESSED_PATH) / 1024:.1f} KB')